In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install torch torchvision

import torch
print(f'PyTorch version: {torch.__version__}')
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

PyTorch version: 2.10.0+cu128
GPU available: True
GPU: Tesla T4


In [3]:
import os
import torch
from PIL import Image

# ✏️ UPDATE THESE PATHS
TRAIN_IMAGES = '/content/drive/MyDrive/dataset/images/train'
TRAIN_LABELS = '/content/drive/MyDrive/dataset/labels/train'
VAL_IMAGES   = '/content/drive/MyDrive/dataset/images/val'
VAL_LABELS   = '/content/drive/MyDrive/dataset/labels/val'

# ✏️ UPDATE CLASSES — must match your dataset.yaml order
# Index 0 is always background for Faster RCNN
CLASSES = ['background', 'blue_boost', 'pepsi', 'purple_boost', 'quavers', 'red_boost', 'tuna_sandwhich']
NUM_CLASSES = len(CLASSES)  # 7 (6 products + background)

print(f'Classes: {CLASSES}')
print(f'Number of classes: {NUM_CLASSES}')

Classes: ['background', 'blue_boost', 'pepsi', 'purple_boost', 'quavers', 'red_boost', 'tuna_sandwhich']
Number of classes: 7


In [4]:
def convert_yolo_to_rcnn(image_folder, label_folder):
    dataset = []
    skipped = 0

    for img_name in os.listdir(image_folder):
        if not img_name.lower().endswith('.jpg'):
            continue

        img_path = os.path.join(image_folder, img_name)
        label_path = os.path.join(label_folder, img_name.replace('.jpg', '.txt').replace('.JPG', '.txt'))

        if not os.path.exists(label_path):
            skipped += 1
            continue

        img = Image.open(img_path)
        w, h = img.size

        boxes = []
        labels = []

        with open(label_path, 'r') as f:
            for line in f.readlines():
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                class_id = int(parts[0])
                x_center = float(parts[1]) * w
                y_center = float(parts[2]) * h
                width    = float(parts[3]) * w
                height   = float(parts[4]) * h

                x1 = x_center - width / 2
                y1 = y_center - height / 2
                x2 = x_center + width / 2
                y2 = y_center + height / 2

                boxes.append([x1, y1, x2, y2])
                labels.append(class_id + 1)  # +1 because 0 = background

        if len(boxes) == 0:
            skipped += 1
            continue

        dataset.append({
            'image_path': img_path,
            'boxes': torch.tensor(boxes, dtype=torch.float32),
            'labels': torch.tensor(labels, dtype=torch.int64)
        })

    print(f'Loaded: {len(dataset)} images | Skipped: {skipped}')
    return dataset


train_data = convert_yolo_to_rcnn(TRAIN_IMAGES, TRAIN_LABELS)
val_data   = convert_yolo_to_rcnn(VAL_IMAGES, VAL_LABELS)

print(f'\nTrain samples: {len(train_data)}')
print(f'Val samples:   {len(val_data)}')

Loaded: 600 images | Skipped: 0
Loaded: 144 images | Skipped: 0

Train samples: 600
Val samples:   144


In [5]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

class ProductDataset(Dataset):
    def __init__(self, data):
        self.data = data
        self.transforms = transforms.Compose([transforms.ToTensor()])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        img = Image.open(item['image_path']).convert('RGB')
        img = self.transforms(img)
        target = {
            'boxes':  item['boxes'],
            'labels': item['labels']
        }
        return img, target

def collate_fn(batch):
    return tuple(zip(*batch))

train_loader = DataLoader(ProductDataset(train_data), batch_size=4, shuffle=True,  collate_fn=collate_fn)
val_loader   = DataLoader(ProductDataset(val_data),   batch_size=4, shuffle=False, collate_fn=collate_fn)

print(f'Train batches: {len(train_loader)}')
print(f'Val batches:   {len(val_loader)}')

Train batches: 150
Val batches:   36


In [6]:
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

def get_model(num_classes):
    model = fasterrcnn_resnet50_fpn(pretrained=True)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = get_model(NUM_CLASSES)
model.to(device)

print(f'Model loaded on: {device}')

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=FasterRCNN_ResNet50_FPN_Weights.COCO_V1`. You can also use `weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth


100%|██████████| 160M/160M [00:01<00:00, 130MB/s]


Model loaded on: cuda


In [ ]:
NUM_EPOCHS = 20

optimizer = torch.optim.SGD(model.parameters(), lr=0.005, momentum=0.9, weight_decay=0.0005)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

best_loss = float('inf')

for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0

    for images, targets in train_loader:
        images  = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses    = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        total_loss += losses.item()

    scheduler.step()
    avg_loss = total_loss / len(train_loader)
    print(f'Epoch [{epoch+1}/{NUM_EPOCHS}] - Loss: {avg_loss:.4f}')

    # Save best model
    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model.state_dict(), '/content/drive/MyDrive/faster_rcnn_best.pth')
        print(f'  ✅ Best model saved (loss: {best_loss:.4f})')

print('\nTraining complete!')

Epoch [1/20] - Loss: 0.1876
  ✅ Best model saved (loss: 0.1876)
Epoch [2/20] - Loss: 0.0840
  ✅ Best model saved (loss: 0.0840)
Epoch [3/20] - Loss: 0.0518
  ✅ Best model saved (loss: 0.0518)
Epoch [4/20] - Loss: 0.0387
  ✅ Best model saved (loss: 0.0387)
Epoch [5/20] - Loss: 0.0311
  ✅ Best model saved (loss: 0.0311)
Epoch [6/20] - Loss: 0.0294
  ✅ Best model saved (loss: 0.0294)
Epoch [7/20] - Loss: 0.0248
  ✅ Best model saved (loss: 0.0248)
Epoch [8/20] - Loss: 0.0249
Epoch [9/20] - Loss: 0.0225
  ✅ Best model saved (loss: 0.0225)
Epoch [10/20] - Loss: 0.0212
  ✅ Best model saved (loss: 0.0212)
Epoch [11/20] - Loss: 0.0170
  ✅ Best model saved (loss: 0.0170)
Epoch [12/20] - Loss: 0.0156
  ✅ Best model saved (loss: 0.0156)
Epoch [13/20] - Loss: 0.0151
  ✅ Best model saved (loss: 0.0151)
Epoch [14/20] - Loss: 0.0147
  ✅ Best model saved (loss: 0.0147)
Epoch [15/20] - Loss: 0.0145
  ✅ Best model saved (loss: 0.0145)
Epoch [16/20] - Loss: 0.0143
  ✅ Best model saved (loss: 0.0143)
Epoch